In [1]:
import os

In [2]:
%pwd

'd:\\MachineLearning\\END-TO-ENT-MACHINE-LEARNING-PROJECT-WITH-MLFLOW\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\MachineLearning\\END-TO-ENT-MACHINE-LEARNING-PROJECT-WITH-MLFLOW'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path 

In [6]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = Path(config.root_dir),
            source_URL = config.source_URL,
            local_data_file = Path(config.local_data_file),
            unzip_dir = Path(config.unzip_dir)
        )
        return data_ingestion_config

In [8]:
import os 
import urllib.request as request
import zipfile
from  mlProject import logger
from mlProject.utils.common import get_size


In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")


    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the datadirectory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)

In [12]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-11-23 12:47:41,891: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-11-23 12:47:41,894: INFO: common: yaml file: params.yaml loaded successfully]
[2026-11-23 12:47:41,895: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-11-23 12:47:41,896: INFO: common: created directory at: artifacts]
[2026-11-23 12:47:41,899: INFO: common: created directory at: artifacts/data_ingestion]
[2026-11-23 12:47:41,900: INFO: 417685085: File already exists of size: ~ 23 KB]
